In [1]:
"""Ячейка 1 — импорты и переменные окружения.

`load_dotenv()` подтягивает `LLM_API_KEY` из `.env`, без него RAGAS-метрики
(Ячейка 5) не смогут поднять LLM-клиент. `json` и `Path` нужны только в
Ячейке 6 для записи итогового файла с метриками.
"""
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

os.environ.setdefault("NO_PROXY", "localhost,127.0.0.1,qdrant")
os.environ.setdefault("no_proxy", os.environ["NO_PROXY"])
os.environ.setdefault("HTTP_PROXY", "http://127.0.0.1:10809")
os.environ.setdefault("HTTPS_PROXY", "http://127.0.0.1:10809")

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

True

In [2]:
"""Ячейка 2 — golden-датасет из 10 эталонных вопросов.

Структура: 7 английских вопросов по 3 темам корпуса (linear_model / tree
/ model_evaluation), 2 русских для проверки мультиязычного retrieval'а,
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""
GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [ ]:
"""Ячейка 3 — прямое подключение к RAG-pipeline без HTTP-слоя.

`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в этом же процессе. Прямой вызов даёт чистые тайминги и доступ к
`retrieved_contexts`, которые понадобятся RAGAS в Ячейке 5.
"""
from app.config import settings
from app.rag.chain import build_rag_chain
chain, retriever = build_rag_chain()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
"""Ячейка 4 — Recall@k для retriever'а.

Прогоняем каждый golden-вопрос через retriever, забираем URL'ы топ-k чанков
и считаем hit по keyword-совпадению. Дополнительно копим
`retrieved_contexts` — они уйдут в RAGAS на следующем шаге.
"""
def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.

    Используется в Recall@k: keyword-matching на уровне модуля sklearn
    (`linear_model` / `tree` / `model_evaluation`) — этого достаточно
    для понимания «не промазал ли retriever мимо темы целиком».
    """
    if not expected_keywords:  # ловушка под OOC-вопросы, если решите добавлять
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 1.000  (10/10)


In [5]:
"""Ячейка 5 — generation-метрики через RAGAS.

RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`). LLM
и эмбеддер оборачиваем в `LangchainLLMWrapper` и
`LangchainEmbeddingsWrapper` — без этого `evaluate()` не поймёт, как
через них ходить. Считаем Faithfulness (опора на контекст) и
ResponseRelevancy (релевантность ответа вопросу).
"""
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

from app.llm import get_llm

# Прогоняем chain и собираем ответы
for r in results:
    r["response"] = chain.invoke(r["question"])

# RAGAS 0.2.x требует именно эти имена полей в Dataset:
# user_input / response / retrieved_contexts / reference
ragas_data = Dataset.from_list([
    {
        "user_input": r["question"],
        "response": r["response"],
        "retrieved_contexts": r["retrieved_contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

# LLM и эмбеддер для RAGAS оборачиваются в специальные wrapper'ы
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)

ragas_scores = evaluate(
    dataset=ragas_data,
    metrics=[
        Faithfulness(llm=llm),
        ResponseRelevancy(llm=llm, embeddings=emb),
    ],
)
print(ragas_scores)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Exception raised in Job[10]: OutputParserException(Invalid json output: To analyze the complexity of each sentence in the answer and break it down into fully understandable statements without using pronouns, we follow these steps:\n\n1. Identify the key components of the answer.\n2. Break down the answer into simple, understandable statements.\n3. Ensure each statement does not contain pronouns.\n\nGiven the input: {\n    \"question\": \"What is the formula for precision?\",\n    \"answer\": \"The formula for precision is: \\n\\n\\\\[\\\\text{precision} = \\frac{\\\\text{tp}}{\\\\text{tp} + \\\\text{fp}}\\\\]\\\n\\n[2]\"\n}\n\nThe answer contains a formula for precision, which is precision = tp / (tp + fp). This formula can be described in a straightforward statement without using pronouns.\n\nOutput: {\n    \"statements\": [\n        \"The formula for precision is calculated as the number of true positives divided by the sum of true positives and false positives.\"\n    ]\n}
For troub

{'faithfulness': 0.8269, 'answer_relevancy': 0.8473}


In [6]:
"""Ячейка 6 — сохраняем сводку метрик в JSON.

Берём mean по каждой RAGAS-метрике, добавляем retriever-Recall@k и
метаданные прогона (модель, embedder, top_k). Файл `rag_metrics.json`
цитируется в README + входит в будущую легенду на собесе.
"""
# RAGAS 0.2.x возвращает EvaluationResult — берём .to_pandas() для агрегации
df = ragas_scores.to_pandas()
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}
Path("notebooks/rag_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False)
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "n_questions": 10,
  "model": "meta-llama/llama-3.3-70b-instruct",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 10
  },
  "generation": {
    "faithfulness": 0.8269179894179893,
    "answer_relevancy": 0.8472571023268601
  }
}
